In [8]:
import os
import json
import random
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, hamming_loss, accuracy_score,
    classification_report
)
from sklearn.model_selection import RandomizedSearchCV
from funcs import *

In [9]:
def load_dataset(n_per_label=len(CHOSEN_LABELS)):
    """
    Returns:
        texts  : list of strings (prob_desc_description)
        labels : list of sets   (subset of LABELS)
    One entry per unique file. A file appearing in multiple labels
    is loaded once and carries all its matched labels.
    """
    file_to_labels = {}   # fname -> set of matched labels

    for label in CHOSEN_LABELS:
        for fname in fetch_files_per_label(label, n_per_label):
            file_to_labels.setdefault(fname, set()).add(label)

    texts, labels = [], []
    for fname, lbls in file_to_labels.items():
        path = os.path.join(DATA_DIR, fname)
        with open(path, encoding="utf-8") as f:
            rec = json.load(f)
        text = rec.get("prob_desc_description", "") or ""
        if text.strip():
            texts.append(text)
            labels.append(lbls)

    return texts, labels


# ── 2. binarize labels ────────────────────────────────────────────────────
def binarize(labels):
    mlb = MultiLabelBinarizer(classes=CHOSEN_LABELS)
    Y   = mlb.fit_transform(labels)
    return Y, mlb


# ── 3. split ──────────────────────────────────────────────────────────────
def split(texts, Y, test_size=0.2, val_size=0.1, seed=42):
    """
    Split at the sample level (each sample = one unique file).
    We use random split here; for strict label-stratified split
    install skmultilearn and swap in iterative_train_test_split.
    """
    n = len(texts)
    idx = list(range(n))
    random.seed(seed)
    random.shuffle(idx)

    n_test = int(n * test_size)
    n_val  = int(n * val_size)

    test_idx  = idx[:n_test]
    val_idx   = idx[n_test:n_test + n_val]
    train_idx = idx[n_test + n_val:]

    def subset(indices):
        return [texts[i] for i in indices], Y[indices]

    return subset(train_idx), subset(val_idx), subset(test_idx)


# ── 4. TF-IDF ─────────────────────────────────────────────────────────────
def build_tfidf(train_texts):
    vec = TfidfVectorizer(
        ngram_range=(1, 2),   # unigrams + bigrams
        max_features=30_000,
        sublinear_tf=True,    # apply log(1+tf) — helps with long texts
        strip_accents="unicode",
        min_df=2,             # ignore terms appearing in < 2 docs
    )
    X_train = vec.fit_transform(train_texts)
    return vec, X_train


# ── 5. model ──────────────────────────────────────────────────────────────
def build_model():
    return OneVsRestClassifier(
        LogisticRegression(
            max_iter=1000,
            C=1.0,                    # inverse regularization strength
            class_weight="balanced",  # reweights for imbalanced labels
            solver="lbfgs",
        ),
        n_jobs=-1   # fit all binary classifiers in parallel
    )


# ── 6. metrics ────────────────────────────────────────────────────────────
def evaluate(model, X, Y_true, split_name="test"):
    Y_pred = model.predict(X)

    micro_f1  = f1_score(Y_true, Y_pred, average="micro",    zero_division=0)
    macro_f1  = f1_score(Y_true, Y_pred, average="macro",    zero_division=0)
    hloss     = hamming_loss(Y_true, Y_pred)
    subset_acc = accuracy_score(Y_true, Y_pred)   # exact match

    print(f"\n── {split_name} metrics ───────────────────────────────────")
    print(f"  Micro  F1      : {micro_f1:.4f}  (global — dominated by frequent labels)")
    print(f"  Macro  F1      : {macro_f1:.4f}  (avg per label — penalises rare failures)")
    print(f"  Hamming loss   : {hloss:.4f}  (fraction of wrong label bits, lower=better)")
    print(f"  Subset accuracy: {subset_acc:.4f}  (exact full-label match, strictest)")

    print(f"\n── per-label F1 ({split_name}) ────────────────────────────")
    print(classification_report(Y_true, Y_pred, target_names=CHOSEN_LABELS, zero_division=0))

    return {"micro_f1": micro_f1, "macro_f1": macro_f1,
            "hamming": hloss, "subset_acc": subset_acc}


# ── 7. threshold tuning ───────────────────────────────────────────────────
def tune_thresholds(model, X_val, Y_val, steps=20):
    """
    For each label, find the threshold on predicted probabilities
    that maximises F1 on the validation set.
    Returns an array of per-label thresholds.
    """
    probs      = model.predict_proba(X_val)
    thresholds = np.linspace(0.1, 0.9, steps)
    best_t     = np.full(len(CHOSEN_LABELS), 0.5)

    for j in range(len(CHOSEN_LABELS)):
        best_f1 = 0
        for t in thresholds:
            preds = (probs[:, j] >= t).astype(int)
            f1    = f1_score(Y_val[:, j], preds, zero_division=0)
            if f1 > best_f1:
                best_f1  = f1
                best_t[j] = t
        print(f"  {CHOSEN_LABELS[j]:<18} best threshold = {best_t[j]:.2f}  (F1={best_f1:.3f})")

    return best_t


def predict_with_thresholds(model, X, thresholds):
    probs = model.predict_proba(X)
    return (probs >= thresholds).astype(int)



In [ ]:
print("Loading dataset...")
texts, labels = load_dataset()
print(f"  {len(texts)} unique files loaded")

Y, mlb = binarize(labels)
(train_texts, Y_train), (val_texts, Y_val), (test_texts, Y_test) = split(texts, Y)
print(f"  train={len(train_texts)}  val={len(val_texts)}  test={len(test_texts)}")

vec, X_train = build_tfidf(train_texts)
X_val        = vec.transform(val_texts)
X_test       = vec.transform(test_texts)

Loading dataset...
  2678 unique files loaded
  train=1876  val=267  test=535

Fitting TF-IDF...


In [13]:
def print_label_stats(Y_train, Y_val, Y_test, label_names):
    print(f"\n{'Label':<25} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
    print("-" * 60)
    for k, label in enumerate(label_names):
        tr = Y_train[:, k].sum()
        va = Y_val[:,   k].sum()
        te = Y_test[:,  k].sum()
        total = tr + va + te
        print(f"{label:<25}"
              f"{tr/total*100:>7.1f}%"
              f"{va/total*100:>7.1f}%"
              f"{te/total*100:>7.1f}%"
              f"{int(total):>8}")
    print("-" * 60)
    
print_label_stats(Y_train, Y_val, Y_test, CHOSEN_LABELS)


Label                        Train      Val     Test    Total
------------------------------------------------------------
math                        70.8%    9.0%   20.2%    1408
graphs                      69.2%   10.5%   20.3%     542
strings                     70.9%   10.7%   18.5%     422
number theory               72.6%    9.4%   18.0%     350
trees                       70.7%    9.9%   19.4%     324
geometry                    72.9%    7.2%   19.9%     166
games                       77.1%   11.4%   11.4%     105
probabilities               76.1%    8.7%   15.2%      92
------------------------------------------------------------


In [ ]:
print("Training OneVsRest LogisticRegression...")
model = build_model()
model.fit(X_train, Y_train)

# evaluate with default threshold (0.5)
evaluate(model, X_val,  Y_val,  split_name="validation")
evaluate(model, X_test, Y_test, split_name="test")

# tune thresholds on val, re-evaluate on test
print("\nTuning per-label thresholds on validation set...")
thresholds = tune_thresholds(model, X_val, Y_val)

print("\n── test metrics after threshold tuning ──────────────────────")
Y_pred_tuned = predict_with_thresholds(model, X_test, thresholds)
print(f"  Micro  F1      : {f1_score(Y_test, Y_pred_tuned, average='micro',  zero_division=0):.4f}")
print(f"  Macro  F1      : {f1_score(Y_test, Y_pred_tuned, average='macro',  zero_division=0):.4f}")
print(f"  Hamming Score   : {1-hamming_loss(Y_test, Y_pred_tuned):.4f}")

print(classification_report(Y_pred_tuned, Y_test, target_names=CHOSEN_LABELS, zero_division=0))

Training OneVsRest LogisticRegression...

── validation metrics ───────────────────────────────────
  Micro  F1      : 0.7209  (global — dominated by frequent labels)
  Macro  F1      : 0.6983  (avg per label — penalises rare failures)
  Hamming loss   : 0.0899  (fraction of wrong label bits, lower=better)
  Subset accuracy: 0.4831  (exact full-label match, strictest)

── per-label F1 (validation) ────────────────────────────
               precision    recall  f1-score   support

         math       0.74      0.79      0.76       127
       graphs       0.65      0.72      0.68        57
      strings       0.76      0.76      0.76        45
number theory       0.54      0.76      0.63        33
        trees       0.62      0.78      0.69        32
     geometry       0.56      0.75      0.64        12
        games       0.92      0.92      0.92        12
probabilities       0.75      0.38      0.50         8

    micro avg       0.69      0.76      0.72       326
    macro avg     

### Random-Forest & GridSearch :

In [15]:
model = OneVsRestClassifier(
        RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",   # handles label imbalance
            n_jobs=-1,                 # use all CPU cores
            random_state=42
        )
    )


param_grids = {
        "estimator__n_estimators":   [100, 300, 500],
        "estimator__max_depth":      [10, 20],
        "estimator__min_samples_leaf": [1, 2, 5],
        "estimator__max_features":   ["sqrt", "log2"]
    }




In [16]:
search = RandomizedSearchCV(
    estimator           = model,
    param_distributions = param_grids,  
    n_iter              = 20,
    cv                  = 5,
    scoring             = "f1_macro",
    n_jobs              = -1,
    random_state        = 42,
    verbose             = 1
)
search.fit(X_train, Y_train)

print(f"Best params : {search.best_params_}")
print(f"Best val F1 : {search.best_score_:.4f}")

# replace model with the best estimator found
model = search.best_estimator_



Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params : {'estimator__n_estimators': 500, 'estimator__min_samples_leaf': 2, 'estimator__max_features': 'sqrt', 'estimator__max_depth': 10}
Best val F1 : 0.6717


In [30]:
model.fit(X_train, Y_train)

# evaluate with default threshold (0.5)
# evaluate(model, X_val,  Y_val,  split_name="validation")
evaluate(model, X_test, Y_test, split_name="test")

# tune thresholds on val, re-evaluate on test
print("\nTuning per-label thresholds on validation set...")
thresholds = tune_thresholds(model, X_val, Y_val)

print("\n── test metrics after threshold tuning ──────────────────────")
Y_pred_tuned = predict_with_thresholds(model, X_test, thresholds)

print(f"  Micro  F1      : {f1_score(Y_test, Y_pred_tuned, average='micro',  zero_division=0):.4f}")
print(f"  Macro  F1      : {f1_score(Y_test, Y_pred_tuned, average='macro',  zero_division=0):.4f}")
print(f"  Hamming Score   : {1-hamming_loss(Y_test, Y_pred_tuned):.4f}")


── test metrics ───────────────────────────────────
  Micro  F1      : 0.7270  (global — dominated by frequent labels)
  Macro  F1      : 0.6561  (avg per label — penalises rare failures)
  Hamming loss   : 0.0881  (fraction of wrong label bits, lower=better)
  Subset accuracy: 0.4916  (exact full-label match, strictest)

── per-label F1 (test) ────────────────────────────
               precision    recall  f1-score   support

         math       0.80      0.84      0.82       284
       graphs       0.61      0.71      0.66       110
      strings       0.76      0.78      0.77        78
number theory       0.51      0.65      0.57        63
        trees       0.57      0.70      0.63        63
     geometry       0.65      0.79      0.71        33
        games       0.65      0.92      0.76        12
probabilities       0.75      0.21      0.33        14

    micro avg       0.69      0.76      0.73       657
    macro avg       0.66      0.70      0.66       657
 weighted avg   

In [36]:
print(classification_report(Y_pred_tuned, Y_test, target_names=CHOSEN_LABELS, zero_division=0))

               precision    recall  f1-score   support

         math       0.92      0.75      0.83       348
       graphs       0.73      0.58      0.65       137
      strings       0.83      0.74      0.78        88
number theory       0.51      0.73      0.60        44
        trees       0.49      0.82      0.61        38
     geometry       0.64      0.72      0.68        29
        games       0.75      0.75      0.75        12
probabilities       0.71      0.33      0.45        30

    micro avg       0.77      0.70      0.74       726
    macro avg       0.70      0.68      0.67       726
 weighted avg       0.80      0.70      0.74       726
  samples avg       0.80      0.74      0.74       726



## Conclusion : 
> best model so far : random forest with parameters : {'estimator__n_estimators': 500, 'estimator__min_samples_leaf': 2, 'estimator__max_features': 'sqrt', 'estimator__max_depth': 10}

> to store and will be used in the CLI.

In [ ]:
import joblib
import json
import numpy as np

# # best model from search
best_model = search.best_estimator_
FOLDER = "best_model_All/"

# # save model
joblib.dump(best_model, FOLDER+"best_rf_model.pkl")

# # save thresholds (assuming thresholds is a dict {label_name: float})
# # thresholds is already a dict {label_name: float} from tune_thresholds
with open("thresholds.json", "w") as f:
    json.dump({k: float(v) for k, v in zip(CHOSEN_LABELS,thresholds)}, f, indent=2)

# saving the vectorier to ensure model sees same feature space.
joblib.dump(vec, FOLDER+"vectorizer.pkl")


Saved: best_rf_model.pkl, thresholds.json, label_names.json
